# ASSIGNMENT_01

In [ ]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed classes

In [31]:
from topologicpy.Vertex import Vertex
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper


## 2. Check the TopologicPy version

In [32]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.20) is OLDER than the latest version (0.9.21) from PyPI. Please consider upgrading to the latest version.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [33]:
renderer = "vscode"

## 3. Getting OBJ file

In [52]:
objects = Topology.ByOBJPath(r"C:\Users\zeyne\OneDrive\Masaüstü\GitHubSaves\GML_MACAD\Assignment_01\.venv\house.obj")
print(objects)


[<topologic_core.Cluster object at 0x000002C16ACDC3B0>, <topologic_core.Cluster object at 0x000002C16ACDDCB0>, <topologic_core.Cluster object at 0x000002C16ACDC370>, <topologic_core.Cluster object at 0x000002C16AD5C770>]


In [53]:
cells = []
selectors = []
for object in objects:
    d = Topology.Dictionary(object)
    faces = Topology.Faces(object)
    if len(faces) >1:
        c = Cell.ByFaces(faces)
        c = Topology.RemoveCollinearEdges(c)
        s = Topology.InternalVertex(c)
        name = Dictionary.ValueAtKey(d, "name")
        if "Room_" in name:
            color = "blue"
        d = Dictionary.SetValuesAtKeys(d, ["color", "vertex_size"], [color, 20])
        s = Topology.SetDictionary(s, d)
        selectors.append(s)
        cells.append(c)
        print(Dictionary.Keys(d), Dictionary.Values(d))

print(len(cells))

['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['blue', 'Room_01', '', 'Room_01', 1.0, 20]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['blue', 'Room_02', '', 'Room_02', 1.0, 20]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['blue', 'Room_03', '', 'Room_03', 1.0, 20]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['blue', 'Room_04', '', 'Room_04', 1.0, 20]
4


## 4. Forming the cells

In [54]:
house = CellComplex.ByCells(cells)
house = Topology.TransferDictionariesBySelectors(house, selectors, tranCells= True)
house_cells = Topology.Cells(house)
for house_cell in house_cells:
    d = Topology.Dictionary(house_cell)
    print(Dictionary.Keys(d), Dictionary.Values(d))

['aabb', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[0.0, 0.0, 0.0, 4.318279, 4.118762, 3.0], 'blue', 'Room_01', '', 'Room_01', 1.0, 20]
['aabb', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[4.318279, 0.0, 0.0, 10.0, 3.211073, 3.0], 'blue', 'Room_04', '', 'Room_04', 1.0, 20]
['aabb', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[4.318279, 3.211073, 0.0, 10.0, 8.0, 3.0], 'blue', 'Room_03', '', 'Room_03', 1.0, 20]
['aabb', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[0.0, 4.118762, 0.0, 4.318279, 8.0, 3.0], 'blue', 'Room_02', '', 'Room_02', 1.0, 20]


In [55]:
Topology.Show(house_cells, faceColorKey= "color", faceOpacity = 0.8, opacityKey="nothing", renderer= renderer)

## 5. Vertex Graph

In [56]:
g = Graph.ByTopology(house)
verts = Graph.Vertices(g)
for vert in verts:
    d = Topology.Dictionary(vert)
    print(Dictionary.Keys(d), Dictionary.Values(d))

['aabb', 'category', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[0.0, 0.0, 0.0, 4.318279, 4.118762, 3.0], 0, 'blue', 'Room_01', '', 'Room_01', 1.0, 20]
['aabb', 'category', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[4.318279, 3.211073, 0.0, 10.0, 8.0, 3.0], 0, 'blue', 'Room_03', '', 'Room_03', 1.0, 20]
['aabb', 'category', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[4.318279, 0.0, 0.0, 10.0, 3.211073, 3.0], 0, 'blue', 'Room_04', '', 'Room_04', 1.0, 20]
['aabb', 'category', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[0.0, 4.118762, 0.0, 4.318279, 8.0, 3.0], 0, 'blue', 'Room_02', '', 'Room_02', 1.0, 20]


In [57]:
Topology.Show(g, house, vertexColorKey= "color", vertexSizeKey= "vertex_size", renderer= renderer)

## 6. Adding Apertures

In [73]:
apertures = []
object = Topology.ByOBJPath(r"C:\Users\zeyne\OneDrive\Masaüstü\GitHubSaves\GML_MACAD\Assignment_01\.venv\windows.obj")
faces = [Topology.Faces(object)[0] for object in object]
for face in faces:
    face = Topology.RemoveCollinearEdges(face)
    d = Dictionary.ByKeysValues(["type", "color", "vertex_size"], ["window", "pink", 20])
    face = Topology.SetDictionary(face, d)
    apertures.append(face)
print(apertures)
print(len(apertures))


[<topologic_core.Face object at 0x000002C1678E22B0>, <topologic_core.Face object at 0x000002C16AD87B30>, <topologic_core.Face object at 0x000002C16AB9FE30>, <topologic_core.Face object at 0x000002C16788FCF0>, <topologic_core.Face object at 0x000002C16825CFB0>, <topologic_core.Face object at 0x000002C16AD942F0>]
6


In [74]:
object = Topology.ByOBJPath(r"C:\Users\zeyne\OneDrive\Masaüstü\GitHubSaves\GML_MACAD\Assignment_01\.venv\doors.obj")
faces = [Topology.Faces(object)[0] for object in object]
for face in faces:
    face = Topology.RemoveCollinearEdges(face)
    d = Dictionary.ByKeysValues(["type", "color", "vertex_size"], ["door", "purple", 20])
    face = Topology.SetDictionary(face, d)
    apertures.append(face)
print(apertures)
print(len(apertures))

[<topologic_core.Face object at 0x000002C1678E22B0>, <topologic_core.Face object at 0x000002C16AD87B30>, <topologic_core.Face object at 0x000002C16AB9FE30>, <topologic_core.Face object at 0x000002C16788FCF0>, <topologic_core.Face object at 0x000002C16825CFB0>, <topologic_core.Face object at 0x000002C16AD942F0>, <topologic_core.Face object at 0x000002C16AC35D70>, <topologic_core.Face object at 0x000002C168D62D30>, <topologic_core.Face object at 0x000002C168D12D30>]
9


In [75]:
house = Topology.AddApertures(house, apertures, subTopologyType= "face")

In [76]:
g= Graph.ByTopology(house, direct = False, viaSharedApertures= True, toExteriorApertures= True)
verts = Graph.Vertices(g)
for v in verts:
    d = Topology.Dictionary(v)
    print(Dictionary.Keys(d),Dictionary.Values(d))

['category', 'color', 'type', 'vertex_size'] [4, 'pink', 'Aperture', 20]
['category', 'color', 'type', 'vertex_size'] [4, 'pink', 'Aperture', 20]
['aabb', 'category', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[0.0, 4.118762, 0.0, 4.318279, 8.0, 3.0], 0, 'blue', 'Room_02', '', 'Room_02', 1.0, 20]
['category', 'color', 'type', 'vertex_size'] [2, 'purple', 'Aperture', 20]
['aabb', 'category', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[4.318279, 0.0, 0.0, 10.0, 3.211073, 3.0], 0, 'blue', 'Room_04', '', 'Room_04', 1.0, 20]
['aabb', 'category', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[4.318279, 3.211073, 0.0, 10.0, 8.0, 3.0], 0, 'blue', 'Room_03', '', 'Room_03', 1.0, 20]
['aabb', 'category', 'color', 'group', 'material', 'name', 'opacity', 'vertex_size'] [[0.0, 0.0, 0.0, 4.318279, 4.118762, 3.0], 0, 'blue', 'Room_01', '', 'Room_01', 1.0, 20]
['category', 'color', 'type', 'vertex_size'] [2, 'purple', 'Aperture', 20]
['cate

In [77]:
Topology.Show(house, apertures, vertexColorKey= "color", vertexSizeKey= "vertex_size", backgroundColor= "white", renderer= renderer)

In [79]:
Topology.Show(g, house, apertures, vertexColorKey= "color", vertexSizeKey= "vertex_size", backgroundColor= "white", renderer= renderer)

# 7. Adding edges and vertices

In [88]:
vertices = Graph.Vertices(g)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color", "vertex_size"], [18, "brown", 20])
    v = Topology.SetDictionary(v, d)
edges = Graph.Edges(g)
for e in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [4, "black"])
    e = Topology.SetDictionary(e, d)

In [89]:
Topology.Show(g, house, vertexColorKey= "color", vertexSizeKey= "vertex_size", renderer= renderer)

# 8. Access Graph

In [82]:
g2 = Graph.ByTopology(house, direct=False, viaSharedApertures=True)

In [100]:
vertices = Graph.Vertices(g2)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color", "vertex_size"], [18, "green", 20])
    v = Topology.SetDictionary(v, d)
edges = Graph.Edges(g2)
for e in edges:
    d = Dictionary.ByKeysValues(["width", "color", "edge_thickness"], [4, "black", 5])
    e = Topology.SetDictionary(e, d)

In [101]:
Topology.Show(g2, house, vertexColorKey= "color", vertexSizeKey= "vertex_size", backgroundColor="lightgray", renderer= renderer)

# 9. Primal Graph

In [106]:
cluster = Cluster.ByTopologies([house,g, g2])
vertices = Topology.Vertices(cluster)
vertices = [Topology.Copy(v) for v in vertices]
edges = Topology.Edges(cluster)
edges = [Topology.Copy(e) for e in edges]
g1 = Graph.ByVerticesEdges(vertices, edges)

In [107]:
vertices = Graph.Vertices(g1)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [18, "red"])
    v = Topology.SetDictionary(v, d)
edges = Graph.Edges(g1)
for e in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [4, "black"])
    e = Topology.SetDictionary(e, d)

In [109]:
Topology.Show(g1, house, vertexColorKey= "color", vertexSizeKey= "vertex_size", renderer= "default")

Plotly.Show - Error: The input renderer is not in the approved list of renderers. Returning None.
